## Waiting & Callbacks

As futures hold a values that will eventually be available, we can either wait for it a specific amountof time, or just leave a callback to be thrown when the future completes.

### Await for a Future

We can bring back syncrhonicity in Futures, excplitly waiting for a future to complete.

In [1]:
import scala.concurrent.Future
import concurrent.ExecutionContext.Implicits.global

import scala.concurrent.duration._
import scala.concurrent.Await
import java.util.concurrent.TimeUnit

import scala.concurrent.Future

import concurrent.ExecutionContext.Implicits.global


import scala.concurrent.duration._

import scala.concurrent.Await

import java.util.concurrent.TimeUnit


In [2]:
val f = Future{Thread.sleep(10000); 4 }
val waitTime = Duration(5, TimeUnit.SECONDS)
val result = Await.result(f, waitTime)

: 

`Await.result` blocks the main thread and waits a defined duration for the result of the given Future. If it’s not complete after that time it throws an exception.

In [3]:
val f = Future{Thread.sleep(4000); 4 }
val result = Await.result(f, 5.seconds)

f: Future[Int] = Success(value = 4)
result: Int = 4

If we wanted to wait forever, we should use `Duration.Inf`

In [ ]:
val result = Await.result(f, Duration.Inf)

`Await.result` blocks the main thread. Only use it when really necessary to wait. 

Non-blocking ways of transforming the Future are generally preferred.

## Callbacks

Given the asynchronous nature of Futures, one of the logical neds is a mechanism to manage callbacks. 
Therefore, when a future is completed, the code can get "notified" and the resulting value of the future can be accessed. 

### onComplete

The `onComplete` method exists for this purpose, and it allows to manage the future value at the completion time:

In [4]:
import scala.util.{Success,Failure}

import scala.util.{Success,Failure}


In [5]:
f.onComplete {
    case t => println(t)
}

The completed feature will be either a `Success` or a `Failure`, so it is possible to directly use cases to distinguish them both.

In [6]:
f onComplete {
    case Success(i) => println(s"successful result: $i")
    case Failure(ex) => println(s"failed: $ex")
}

### foreach

If we want to call a callback only when the Future completes successfully, we should use the `foreach` method.

In [7]:
import scala.util.Random

def giveMeANumber = Future{
    Thread.sleep(4000)
    Random.nextInt(5) 
}

import scala.util.Random


defined function giveMeANumber

In [10]:
giveMeANumber foreach {
    result => println(s"Computed value: ${result*4}")
}
Thread.sleep(5000)

Computed value: 8


This won’t execute the given callback function when the future completes with failure. In the followign example the `foreach` will not be called.

In [11]:
def giveMeABadNumber = Future{
    Thread.sleep(4000)
    throw Exception("bad")
    Random.nextInt(5) 
}

defined function giveMeABadNumber

In [12]:
giveMeABadNumber foreach {
    result => println(s"Computed value: ${result*4}")
}
Thread.sleep(5000)

### andThen

The `andThen` function applies a side-effecting function to the given Future and returns the same Future.

In [ ]:
val f = giveMeANumber andThen {
    case Success(v) => println(s"by the way the value was $v")
} andThen {
    case Success(v) => println(s"we can still do more things with $v")
}

Thread.sleep(5000)